# AF DB Structure Download

Lead : AlexLeonardos

Issue : Github Issue 72 — Download Prefolded AlphaFold Structures NR Sequences

Start : 2026-05-16

Complete : 2026-05-22

Files : ~/igem-toronto/resources/260523_issue72_af_db_download/scripts 

Many folded structures for Uniprot accessions are found on Google Cloud Storage: https://console.cloud.google.com/storage/browser/public-datasets-deepmind-alphafold-v4;tab=objects?prefix=&forceOnObjectsSortingFiltering=false

The purpose of these downloads is to avoid rerunning folding experiments on sequences that have already been folded.

#### System initialization

The conda environment yml is `resources/260523_issue72_af_db_download/af_db_env.yml`.

Fasta File Location:
aws s3 cp s3://petadex/logan/petadex.complete_catalytic_cores.v1.1.fa.zst . - Downloaded May 16th 2026

In [ ]:
# Inspect the first few lines of the file to understand its structure (WSL command):
zstd -dc petadex.complete_catalytic_cores.v1.1.fa.zst | grep "^>" | head -10

Accessions found second:

```
>1|WP_054022242.1|||||/65-261
>2|7NEI|||||/41-230
>3|7QJM|||||/95-371
>4|A0A1W6L588|||||/63-258
>5|ACY95991.1|||||/71-260
>6|ACY96861.1|||||/69-258
>7|ADK73612.1|||||/77-271
>8|ADM47605.1|||||/41-230
>9|ADV66860.1|||||/95-284
>10|ADV92525.1|||||/41-230
```

In [ ]:
# Inspect how many header lines are in the file (WSL command):
zstd -dc petadex.complete_catalytic_cores.v1.1.fa.zst | grep "^>" | wc -l

307155746

Inspect how many are actually valid accessions:

`grep -c "." accessions.csv`

4723339

`awk 'NF' accessions.csv > clean_accessions.csv` - to store all of them without blanks

# Step 1: Make a csv of just the accessions

In [ ]:
# Extract the accession numbers from the file and save them to a new file (WSL command):
zstd -dc petadex.complete_catalytic_cores.v1.1.fa.zst | grep "^>" | sed 's/^>//' | awk -F'[ |]' '{print $2}' > accessions.csv

In [ ]:
# Clean the accessions to remove any blank lines (WSL command):
awk 'NF' accessions.csv > clean_accessions.csv

In [ ]:
# Clean the accessions to remove any version numbers (WSL command):
awk -F'.' '{print $1}' clean_accessions.csv > clean_accessions_no_version.csv

## Step 2: Convert the accessions to Uniprot, if possible using the Uniprot API

Now the accessions have a bunch of different types of accessions. We want to convert them into Uniprot format as that is how they are stored in AlphaFoldDB

Currently they include Uniprot (E9LVH8), GenBank (MBV9193613.1), NCBI RefSeq (WP_041846030.1), and PDB (2ZPQ_A).

`split_accession_types.py` splits these into different csvs, with extras included if they don't fit any regex pattern.

`python3 scripts/split_accession_types.py clean_accessions_no_version.csv`

```
Processing Completed Successfully!
Time Taken: 6.13 seconds
----------------------------------------
Total Rows Parsed:     4,723,339
├── UniProt IDs:       552 -> written to 'uniprot_accessions.csv'
├── RefSeq IDs:        2,605,429 -> written to 'refseq_accessions.csv'
├── GenBank IDs:       2,115,910 -> written to 'genbank_accessions.csv'
└── Unclassified/Other: 1,448 -> written to 'other_accessions.csv'
----------------------------------------
```

After inspection, the Other accessions are all PDB. Many of them are tagged _A, which was stripped as we care about the folding of the individual chains.

Code for converting the accessions to UniprotKB using the Uniprot ID mapping API is found in `accession_mapping.py`.

Usage: `python accession_mapping.py [input_csv] [from_db] [output_tsv] [batch_size]`

Benchmark Speed:
- Instant for PDB (1448)
- Took 53 Minutes for a batch of 100,000 RefSeq (2,605,429)

This was deemed to be too slow, so for step 3 I swapped to a local accession table lookup method.

## Step 3: Local Accession Table Lookup

Downloading the master mapping and analyzing it locally using polars is much faster than the ~ 25 hours it would take to run the API script.

Code is found in `local_accession_mapping.py`.

`pip install polars pyarrow`
`cd resources/260523_issue72_af_db_download/scripts/`
`python local_accession_mapping.py `

Make sure to download `idmapping_selected.tab.gz` from the Uniprot Mapping https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/

# Step 4: Create the Manifest File

This is all of the file names that should exist if they are on AlphaFoldDB.

In [10]:
accessions = []
with open("accessions/mappings/pdb_to_uniprot_test.tsv", "r") as f:
    for line in f:
        # only take the second column which is the uniprot accession, and strip whitespace
        accession = line.strip().split("\t")[1]

        if accession and accession != "Entry":
            accessions.append(accession)

In [11]:
print(len(accessions))
print(accessions[:10])

1502
['A4Y035', 'A0A075B5G4', 'P01267', 'A0A4W2CHS8', 'Q14703', 'Q9H741', 'Q14703', 'E5WZQ7', 'Q869C3', 'Q8NBP7']


In [4]:
with open("accessions/manifests/from_pdb_cif_manifest.txt", "w") as f:
    for uid in accessions:
        f.write(f"gs://public-datasets-deepmind-alphafold-v4/AF-{uid}-F1-model_v4.cif\n")

In [ ]:
# Also make sure to clean up the newlines in the file, I was using WSL
dos2unix accessions/manifests/from_genbank_cif_manifest.txt
dos2unix accessions/manifests/uniprot_cif_manifest.txt
dos2unix accessions/manifests/from_refseq_cif_manifest.txt
dos2unix accessions/manifests/from_pdb_cif_manifest.txt

# Step 5: Run Gcloud to download existing predictions

In [ ]:
# Now use the terminal to run the download command with the manifest file:
cat alphafold_cif_manifest.txt | gsutil -m cp -I af_db/ # this is old, use the new gcloud storage command instead

gcloud storage cp -c -I af_db_refseq/ < accessions/manifests/from_refseq_cif_manifest.txt

Note: This was run on an Azure VM for the space required ~ 169GB.

In [12]:
import os

# Combined list of all expected accessions
all_expected = set(accessions)  # Assuming 'accessions' is the list of expected accessions from the previous step

# Get all filenames in af_db/ and extract accessions
downloaded_files = os.listdir("af_db_pdb/")
downloaded_accessions = set(
    filename.split("-")[1]  # Extract accession from 'AF-<ACCESSION>-F1-model_v4.cif'
    for filename in downloaded_files
    if filename.endswith(".cif")
)

# Find accessions that are missing
missing_accessions = sorted(all_expected - downloaded_accessions)

print(f"Expected: {len(all_expected)}")
print(f"Downloaded: {len(downloaded_accessions)}")
print(f"Missing: {len(missing_accessions)}")

# Save to file
with open("results/missing_pdb_af_accessions.txt", "w") as f:
    for acc in missing_accessions:
        f.write(f"{acc}\n")

Expected: 593
Downloaded: 526
Missing: 67


# Step 6: Upload structures to Azure

**Results:**

*Uniprot:*

Expected: 548

Downloaded: 540

Missing: 8

These are saved in `protein-structures/af_db/genbankrefseq/af_structures` in Azure blob storage. 

*RefSeq + GenBank: (Ran on VM)*

Expected: 830217

Downloaded: 458174

Missing: 372043

These are saved in `protein-structures/af_db/genbankrefseq/af_structures` in Azure blob storage. 

*PDB*

Expected: 593

Downloaded: 526

Missing: 67

These are saved in `protein-structures/af_db/pdb/` in Azure blob storage. 



# Step 7: Figure out what's missing

#### 7A: Identify the Uniprot Accessions that didn't have a fold on af_db.

These are held in the `af_misses` on Azure. They were generated using the code in this notebook.

The files with `true` in the name indicate that these uniprot accessions have been reverse mapped back to their representation in the original FASTA file.


In [ ]:
# Reverse mapping from uniprot to pdb
uniprot_mapping = {}
with open("accessions/mappings/_to_uniprot_test.tsv", "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            pdb_id = parts[0]
            uniprot_id = parts[1]
            uniprot_mapping[uniprot_id] = pdb_id

# missing accessions
with open("af_misses/missing_pdb_af_accessions.txt", "r") as f:
    missing_accessions = [line.strip() for line in f]

# Save the missing PDB IDs to a new file
with open("af_misses/true_pdb_af_misses.txt", "w") as f:
    for uniprot_id in missing_accessions:
        pdb_id = uniprot_mapping.get(uniprot_id)
        if pdb_id:
            f.write(f"{pdb_id}\n")

In [15]:
# Reverse mapping from uniprot to genbank/refseq
uniprot_mapping = {}
with open("accessions/mappings/refseq_to_uniprot.tsv", "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            refseq_id = parts[0]
            uniprot_id = parts[1]
            uniprot_mapping[uniprot_id] = refseq_id

with open("accessions/mappings/genbank_to_uniprot.tsv", "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            genbank_id = parts[0]
            uniprot_id = parts[1]
            uniprot_mapping[uniprot_id] = genbank_id

# missing accessions
with open("af_misses/missing_genbankrefseq_af_accessions.txt", "r") as f:
    missing_accessions = [line.strip() for line in f]

# Save the missing PDB IDs to a new file
with open("af_misses/true_genbankrefseq_af_misses.txt", "w") as f:
    for uniprot_id in missing_accessions:
        id = uniprot_mapping.get(uniprot_id)
        if id:
            f.write(f"{id}\n")


#### 7B: Create a new FASTA with only unfolded proteins.

Iterate through the rows of the original file. 

We only want to disclude rows that let to an AF structure, that is, rows with a Uniprot match and that are not in the af_misses.

set(unfolded) = set(accessions) - set(accessions with a uniprot mapping) + set(af_misses).

In [21]:
import zstandard as zstd
import re

def normalize_accession(acc):
    acc = acc.strip()
    acc = acc.split(".")[0]  # strip version suffix
    acc = re.sub(r"_[A-Za-z]$", "", acc)  # only strip single letter chain suffix e.g. 5VLP_A
    return acc

# 1. Load all accessions that have a UniProt mapping
mapped_accessions = set()
mapping_files = [
    "accessions/mappings/refseq_to_uniprot.tsv",
    "accessions/mappings/genbank_to_uniprot.tsv",
    "accessions/mappings/pdb_to_uniprot_test.tsv"
]

for f_path in mapping_files:
    with open(f_path, "r") as f:
        for line in f:
            token = normalize_accession(line.strip().split("\t")[0])
            if token and token != "From":
                mapped_accessions.add(token)

with open("accessions/uniprot_accessions.csv", "r") as f:
    for line in f:
        token = normalize_accession(line.strip())
        if token:
            mapped_accessions.add(token)

# 2. Load precomputed af_miss_accessions
af_miss_accessions = set()
with open("af_misses/true_genbankrefseq_af_misses.txt", "r") as f:
    for line in f:
        token = normalize_accession(line.strip())
        if token:
            af_miss_accessions.add(token)
with open("af_misses/true_pdb_af_misses.txt", "r") as f:
    for line in f:
        token = normalize_accession(line.strip())
        if token:
            af_miss_accessions.add(token)
with open("af_misses/missing_uniprot_af_accessions.txt", "r") as f:
    for line in f:
        token = normalize_accession(line.strip())
        if token:
            af_miss_accessions.add(token)

# 3. Write output FASTA
dctx = zstd.ZstdDecompressor()
cctx = zstd.ZstdCompressor()

with open("petadex.complete_catalytic_cores.v1.1.fa.zst", "rb") as fin, \
     open("unfolded.fa.zst", "wb") as fout:
    with cctx.stream_writer(fout) as writer:
        buffer = ""
        write = False
        for chunk in dctx.read_to_iter(fin):
            buffer += chunk.decode("utf-8")
            lines = buffer.split("\n")
            # Keep last incomplete line in buffer
            buffer = lines[-1]
            for line in lines[:-1]:
                line = line + "\n"
                if line.startswith(">"):
                    current_accession = normalize_accession(line.split("|")[1])
                    in_mapped = current_accession in mapped_accessions
                    in_af_miss = current_accession in af_miss_accessions
                    write = (not in_mapped) or in_af_miss
                if write:
                    writer.write(line.encode("utf-8"))
        # Handle last line
        if buffer:
            if buffer.startswith(">"):
                current_accession = normalize_accession(buffer.split("|")[1])
                in_mapped = current_accession in mapped_accessions
                in_af_miss = current_accession in af_miss_accessions
                write = (not in_mapped) or in_af_miss
            if write:
                writer.write(buffer.encode("utf-8"))

`unfolded.fa.zst` is also found on Azure (protein-structures/af_db)